In [ ]:
# Import Nav2 class

from nav2_simple_commander.robot_navigator import BasicNavigator
import rclpy

# Initialize: execute again after kernel restart
rclpy.init()
nav = BasicNavigator()

In [ ]:
# Check Nav2 is imported

nav.waitUntilNav2Active()

In [ ]:
# Pinky's angle into Quaternion

import math
import tf_transformations


def get_quaternion_from_yaw(yaw_degrees):
    yaw_radians = math.radians(yaw_degrees)

    quaternion = tf_transformations.quaternion_from_euler(0, 0, yaw_radians)

    return quaternion

In [ ]:
# Create node subscribes to "amcl"
#  to print current location

from geometry_msgs.msg import PoseWithCovarianceStamped

sub_amcl = rclpy.create_node("sub_amcl")


def amcl_callback(msg):
    print("===")
    print(msg.pose.pose.position)
    print(msg.pose.pose.orientation)


sub_amcl.create_subscription(PoseWithCovarianceStamped, "amcl_pose", amcl_callback, 10)

In [ ]:
# Crate Goal
# rotate from current position

from geometry_msgs.msg import PoseStamped

goal_yaw = 180
q = get_quaternion_from_yaw(goal_yaw)

goal_pose = PoseStamped()
goal_pose.header.frame_id = "map"
goal_pose.pose.position.x = 0.5
goal_pose.pose.position.y = 0.0
goal_pose.pose.position.z = 0.0
goal_pose.pose.orientation.x = q[0]
goal_pose.pose.orientation.y = q[1]
goal_pose.pose.orientation.z = q[2]
goal_pose.pose.orientation.w = q[3]

In [ ]:
# send goal to Nav2
# this makes the pinky to change yaw
nav.goToPose(goal_pose)

In [ ]:
# Change yaw and position.x

from geometry_msgs.msg import PoseStamped

goal_yaw = 90
q = get_quaternion_from_yaw(goal_yaw)

goal_pose = PoseStamped()
goal_pose.header.frame_id = "map"
goal_pose.header.stamp = nav.get_clock().now().to_msg()
# make sure this position is valid
goal_pose.pose.position.x = 2.0  # 0.5 -> 2.0
goal_pose.pose.position.y = 0.0
goal_pose.pose.position.z = 0.0
goal_pose.pose.orientation.x = q[0]
goal_pose.pose.orientation.y = q[1]
goal_pose.pose.orientation.z = q[2]
goal_pose.pose.orientation.w = q[3]

In [ ]:
nav.goToPose(goal_pose)

In [ ]:
# while running execute to show feedback

from rclpy.duration import Duration
from IPython.display import clear_output

while not nav.isTaskComplete():
    clear_output(wait=True)
    rclpy.spin_once(sub_amcl, timeout_sec=1.0)

    feedback = nav.getFeedback()
    print("===")
    print(
        "distance remaining: "
        + "{:.2f}".format(feedback.distance_remaining)
        + "meters."
    )

    if Duration.from_msg(feedback.navigation_time) > Duration(seconds=30.0):
        nav.cancelTask()

In [ ]:
# check results

from nav2_simple_commander.robot_navigator import TaskResult

result = nav.getResult()
if result == TaskResult.SUCCEEDED:
    print("Goal succeeded!")
elif result == TaskResult.CANCELED:
    print("Goal was canceled!")
elif result == TaskResult.FAILED:
    print("Goal was failed!")
else:
    print("Goal has an invalid return status!")

In [ ]:
# $ topic echo /clicked_points
# from rviz use "Publish Point" to get the grid value
# grid value will show up in terminal

goal_pose_list = []

goal_yaw1 = 0
goal_pose1 = PoseStamped()
goal_pose1.header.frame_id = "map"
goal_pose1.header.stamp = nav.get_clock().now().to_msg()
# assign x and y value from "Published Point"
goal_pose1.pose.position.x = 0.183
goal_pose1.pose.position.y = 0.434
goal_pose1.pose.orientation.x = q[0]
goal_pose1.pose.orientation.y = q[1]
goal_pose1.pose.orientation.z = q[2]
goal_pose1.pose.orientation.w = q[3]
goal_pose_list.append(goal_pose1)

goal_yaw2 = 180
goal_pose2 = PoseStamped()
goal_pose2.header.frame_id = "map"
goal_pose2.header.stamp = nav.get_clock().now().to_msg()
goal_pose2.pose.position.x = 0.200
goal_pose2.pose.position.y = 1.03
goal_pose2.pose.orientation.x = q[0]
goal_pose2.pose.orientation.y = q[1]
goal_pose2.pose.orientation.z = q[2]
goal_pose2.pose.orientation.w = q[3]
goal_pose_list.append(goal_pose2)

goal_yaw3 = 45
goal_pose3 = PoseStamped()
goal_pose3.header.frame_id = "map"
goal_pose3.header.stamp = nav.get_clock().now().to_msg()
goal_pose3.pose.position.x = 0.733
goal_pose3.pose.position.y = 1.66
goal_pose3.pose.orientation.x = q[0]
goal_pose3.pose.orientation.y = q[1]
goal_pose3.pose.orientation.z = q[2]
goal_pose3.pose.orientation.w = q[3]
goal_pose_list.append(goal_pose3)

In [ ]:
# execute waypoint run

nav_start = nav.get_clock().now()
nav.followWaypoints(goal_pose_list)

In [ ]:
# waypoint run feedback

from rclpy.duration import Duration

while not nav.isTaskComplete():
    clear_output(wait=True)
    rclpy.spin_once(sub_amcl, timeout_sec=1.0)

    feedback = nav.getFeedback()

    print("===")
    print(
        "Executing current waypoint: "
        + str(feedback.current_waypoint + 1)
        + "/"
        + str(len(goal_pose_list))
    )

    now = nav.get_clock().now()

    if now - nav_start > Duration(seconds=360.0):
        navigator.cancelTask()

In [ ]:
# shutdown Nav2 and subscribed node

nav.lifecycleShutdown()
rclpy.shutdown()